In [173]:
import os 
from pathlib import Path
from dotenv import load_dotenv

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd 
load_dotenv(override=True);

In [174]:
data_repo_env = os.getenv('DATA_REPO_PATH')
DATA_REPO_DIR = Path(data_repo_env)

matchescsv = DATA_REPO_DIR / "GW7" / "matches.csv"
teamscsv = DATA_REPO_DIR / "GW7" / "teams.csv"

matches = pd.read_csv(matchescsv)
teams = pd.read_csv(teamscsv)

In [175]:
playerstatscsv = DATA_REPO_DIR / "GW7" / "playermatchstats.csv"
playerscsv = DATA_REPO_DIR / "GW7" / "players.csv"

playermatchstats = pd.read_csv(playerstatscsv)
players = pd.read_csv(playerscsv)

playermatchstats["team_code"] = playermatchstats["player_id"].map(
    players.set_index("player_id")["team_code"]
)

In [176]:
team_sprint = playermatchstats.groupby("team_code").agg(
    total_sprints = ("number_of_sprints", "sum"), 
    avg_distance_cov = ("distance_covered", "mean")
).round(1).reset_index()

In [177]:
teams = teams[["code", "name"]]

In [178]:
metrics = ["total_shots",
           "accurate_passes",
           "shots_inside_box",
           "clearances",
           "duels_won",
           "successful_dribbles"]

home_met = [f"home_{m}" for m in metrics]
away_met = [f"away_{m}" for m in metrics]

home_stats = matches.groupby("home_team")[home_met].sum()
away_stats = matches.groupby("away_team")[away_met].sum()



In [179]:
home_stats.columns = metrics
away_stats.columns = metrics

In [180]:
total_stats = home_stats.add(away_stats, fill_value=0)

In [181]:
teams = teams.merge(
    total_stats,
    left_on = "code",
    right_index=True,
    how="left"
).fillna(0)

In [182]:
teams["number_of_sprints"] = teams["code"].map(
    team_sprint.set_index("team_code")["total_sprints"]
)
teams["avg_distance_cov"] = teams["code"].map(
    team_sprint.set_index("team_code")["avg_distance_cov"]
)

In [189]:
teams_sort_passes = teams.sort_values("accurate_passes").reset_index()
teams_sort_shots_box = teams.sort_values("shots_inside_box").reset_index()
teams_duels_won = teams.sort_values("duels_won").reset_index()
teams_number_of_sprints = teams.sort_values("number_of_sprints").reset_index()
teams_avg_distance_cov = teams.sort_values("avg_distance_cov").reset_index()
